#  Building Resilient OpenSearch Clusters with Cross-Cluster Replication

Import the necessary prerequisites for the poc.

In [ ]:
import boto3
import requests
from requests_aws4auth import AWS4Auth
import os
from dotenv import load_dotenv
import json

Exporting AWS Credentials

In [ ]:
credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    session_token=credentials.token 
)

Load the env variables from the .env

In [ ]:
load_dotenv()
source_host = os.getenv('SOURCE_ENDPOINT')
destination_host = os.getenv('DESTINATION_ENDPOINT')
source_region = os.getenv('SOURCE_REGION')
destination_region = os.getenv('DESTINATION_REGION')
role_arn = os.getenv('ROLE_ARN')
bucket_name = os.getenv('BUCKET_NAME')
repository_name = os.getenv('REPOSITORY_NAME')
snapshot_name = os.getenv('SNAPSHOT_NAME')


Populating the source es !

In [ ]:
logs = [
    {
        "id": "1",
        "body": {
            "timestamp": "2026-03-14T06:00:00Z",
            "level": "INFO",
            "message": "US-East Primary cluster populated",
            "service": "opensearch",
            "region": "us-east-1"
        }
    },
    {
        "id": "2",
        "body": {
            "timestamp": "2026-03-14T06:01:00Z",
            "level": "INFO",
            "message": "Source data ready for CCR to AP-South",
            "service": "opensearch",
            "region": "us-east-1"
        }
    }
]

for log in logs:
    url = f"{source_host}/logs/_doc/{log['id']}"
    response = requests.post(url, json=log['body'])
    print(f"Doc {log['id']} Status: {response.status_code}")

Viewing the populated data !

In [ ]:

endpoint = os.getenv('SOUTH_ENDPOINT', 'http://localhost:9200')
url = f"{endpoint}/logs/_search"

params = {
    "pretty": "true",
    "size": 10
}

try:
    # Execute the GET request
    response = requests.get(url, params=params)
    
    # Check for HTTP errors
    response.raise_for_status()
    
    # Print the search results
    print(json.dumps(response.json(), indent=2))

except requests.exceptions.RequestException as e:
    print(f"Error querying OpenSearch: {e}")


Registering the s3 bucket as a repo in the es for snapshot backup

In [ ]:
def register_opensearch_repository(host, region, repository_name, bucket_name, role_arn):
    """
    Registers an S3 bucket as a snapshot repository in Amazon OpenSearch.
    """
    # 1. Get credentials from the environment (Env vars, ~/.aws/credentials, or IAM Role)
    service = 'es' # Even for OpenSearch, the signing service is 'es'


    # 3. Prepare the request
    url = f"https://{host}/_snapshot/{repository_name}"
    payload = {
        "type": "s3",
        "settings": {
            "bucket": bucket_name,
            "region": region,
            "role_arn": role_arn
        }
    }
    headers = {"Content-Type": "application/json"}

    # 4. Execute the request
    try:
        response = requests.put(url, auth=awsauth, json=payload, headers=headers)
        response.raise_for_status()
        print(f"Successfully registered repository: {repository_name}")
        return response.json()
    
    except requests.exceptions.HTTPError as err:
        print(f"HTTP Error: {err.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None


In [ ]:
register_opensearch_repository(source_host, source_region, repository_name, bucket_name, role_arn)

In [ ]:
register_opensearch_repository(destination_host, destination_region, repository_name, bucket_name, role_arn)

Creating a snapshot and backign up in the s3 bucket.

In [ ]:
url = f"https://{source_host}/_snapshot/{repository_name}/{snapshot_name}?wait_for_completion=true"

print(f"Starting snapshot: {snapshot_name}...")

try:
    response = requests.put(url, auth=awsauth)
    response.raise_for_status()
    print("Success! Snapshot details:")
    print(json.dumps(response.json(), indent=2))
except requests.exceptions.HTTPError as err:
    print(f"Error: {err.response.status_code}")
    print(err.response.text)
except Exception as e:
    print(f"An unexpected error occurred: {e}")


In [ ]:


payload = {
    "indices": "*", 
    "rename_pattern": "(.+)",
    "rename_replacement": "restored_$1"
}

url = f"https://{destination_host}/_snapshot/{repository_name}/{snapshot_name}/_restore"

headers = {"Content-Type": "application/json"}

print(f"Starting restore of {snapshot_name} to {destination_host}...")

# 5. Execute POST request
try:
    response = requests.post(url, auth=awsauth, json=payload, headers=headers)
    response.raise_for_status()
    print("Restore initiated successfully:")
    print(json.dumps(response.json(), indent=2))
except requests.exceptions.HTTPError as err:
    print(f"Error: {err.response.status_code}")
    print(err.response.text)
